# JAAF — monthly apparel exports by market

The Joint Apparel Association Forum is Sri Lanka's apparel industry body. It
publishes **monthly** apparel and textile export values in USD millions, split
by market.

Connector: `ceynex/data/connectors/jaaf.py` (owner: M3 Fernando).

This is CeyNex's only monthly trade series and its only **independent
second source** for apparel — everything else apparel-related comes from
Comtrade or EDB.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. This connector never touches the internet

`srilankaapparel.com` disallows automated fetching in its `robots.txt`. So
`fetch()` makes **no HTTP request at all**. It reads two pages that a person
saved by hand from a browser (Ctrl+S).

That is a deliberate choice to respect the site's stated wishes, and it has one
consequence worth saying out loud:

> **Re-running the connector on a stale saved page silently returns stale data.**
> There is no way for the connector to detect it. Freshness is a human
> responsibility here, not an automated one.

In [2]:
JAAF_DIR = cx.CORE / "data" / "raw" / "jaaf" / "manual"

pages = {
    "annual exports": "Annual Exports – Sri Lanka Apparel.html",
    "market wise": "Market Wise Exports – Sri Lanka Apparel.html",
}

staged = {}
for label, filename in pages.items():
    path = JAAF_DIR / filename
    if path.exists():
        staged[label] = path
        print(f"{label:16} staged  {path.name}  ({path.stat().st_size:,} bytes)")
    else:
        print(f"{label:16} MISSING")

if not staged:
    print(f"\nexpected under: {JAAF_DIR}")
    print("how to get it:  open the JAAF data-center pages in a browser and")
    print("                save each with Ctrl+S under the exact filenames above.")
    print("                Automated fetching is disallowed — do not script it.")

annual exports   MISSING
market wise      MISSING

expected under: /ml/CeyNex/ceynex-core/data/raw/jaaf/manual
how to get it:  open the JAAF data-center pages in a browser and
                save each with Ctrl+S under the exact filenames above.
                Automated fetching is disallowed — do not script it.


## 2. Two page types, two different parsing problems

| Page | Shape | Read with |
|---|---|---|
| Annual Exports | ordinary `<table>` elements | BeautifulSoup |
| Market Wise Exports | **not a table** — values live in a Chart.js `data` array inside a `<script>` | regex |

The second one catches people out. The pie chart on the page looks like data
you could select and copy, but in the HTML it is a JavaScript array. There is
no table to parse.

The market-wise page is used as an **offline cross-check** on the annual
tables, not written to `fact_trade` itself.

## 3. Five tables, only three of them trustworthy

The Annual Exports page has five tables in a fixed order:

| # | Table | Written to `fact_trade`? |
|---|---|---|
| 1 | Total | ✅ as the `partner_iso3 IS NULL` "World" row |
| 2 | US | ✅ `USA` |
| 3 | EU-bloc (approx) | ❌ excluded |
| 4 | UK | ✅ `GBR` |
| 5 | Other (approx) | ❌ excluded |

The tables carry no labels — the order is inferred by cross-matching each
table's latest-year total against the pie chart's named entries. Only Total, US
and UK match a pie-chart label exactly (`US`: 1947.37 vs the pie's 1947.38;
`UK`: 679.66, exact).

**Why the other two are excluded rather than flagged:** written alongside
US/UK/Total they would double-count, exactly as Comtrade's World and EU
aggregate rows would. Brute-force testing which named markets "EU-bloc" and
"Other" would sum to found no coherent geographic grouping — so their
composition is genuinely unknown, not "probably the EU and the rest".

## 4. Two traps confirmed on the real page

**The page prints the literal text `NaN` for some missing cells.** Not blank —
the three characters `N`, `a`, `N`. `float("nan")` parses that happily and does
not raise, so a naive parser writes a `NaN` export value instead of dropping
the row. Seen in the UK table's Feb/Mar cells for several years 2004–2016.

**The current year's unreported months are pre-rendered as `0`, not blank.**
Left unfiltered these were ingested as real observations, which pushed apparel's
latest observed year into the future and corrupted every "what is the most
recent year" query across the whole graph. Real monthly totals run to hundreds
of millions of USD, so an exact zero is never a real observation here.

Both are handled — the first in `_num()`, the second in
`_annual_tables_to_records()`.

## 5. Load

In [3]:
records = None
if "annual exports" in staged:
    from bs4 import BeautifulSoup

    TABLE_LABELS = ["total", "us", "eu_bloc_approx", "uk", "other_approx"]
    MONTHS = {m: i for i, m in enumerate(
        ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], start=1
    )}

    def num(text):
        text = text.replace(",", "").strip()
        if text == "" or text.lower() in ("-", "...", "..", "n/a", "nan"):
            return None
        try:
            return float(text)
        except ValueError:
            return None

    soup = BeautifulSoup(staged["annual exports"].read_text(errors="replace"), "html.parser")
    rows = []
    for i, table in enumerate(soup.find_all("table")):
        trs = table.find_all("tr")
        if not trs:
            continue
        header = [c.get_text(strip=True) for c in trs[0].find_all(["td", "th"])]
        month_cols = [
            (idx, MONTHS[h.strip().lower()[:3]])
            for idx, h in enumerate(header)
            if h.strip().lower()[:3] in MONTHS
        ]
        label = TABLE_LABELS[i] if i < len(TABLE_LABELS) else f"table_{i}"
        for tr in trs[1:]:
            cells = [c.get_text(strip=True) for c in tr.find_all(["td", "th"])]
            if not cells or not cells[0].strip().isdigit():
                continue
            year = int(cells[0].strip())
            for idx, month in month_cols:
                if idx >= len(cells):
                    continue
                value = num(cells[idx])
                if value is None or value == 0:
                    continue
                rows.append({"market": label, "year": year, "month": month, "value_usd_mn": value})

    records = pd.DataFrame(rows)
    print(f"{len(records):,} monthly observations")
    print("markets:", sorted(records["market"].unique()))
    print("years:  ", records["year"].min(), "-", records["year"].max())
    display(records.head())
else:
    print("skipped — no JAAF page staged")

skipped — no JAAF page staged


## 6. Annual totals

Cross-check: JAAF's own market-wise pie gives full-year 2025 apparel and
textile exports of about **USD 5,019.20 million**.

In [4]:
if records is not None and not records.empty:
    totals = (
        records[records["market"] == "total"]
        .groupby("year")["value_usd_mn"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "total_usd_mn", "count": "months_reported"})
    )
    display(totals.tail(12).round(1))

    complete = totals[totals["months_reported"] == 12]
    if not complete.empty:
        ax = complete["total_usd_mn"].plot(
            kind="bar", title="JAAF apparel & textile exports, complete years (USD millions)"
        )
        ax.set_ylabel("USD millions")
        plt.tight_layout()
        plt.show()
    print("\nPartial years are shown above with months_reported < 12 —")
    print("never compare a partial year against a complete one.")
else:
    print("skipped — no data")

skipped — no data


## 7. Seasonality — the reason a monthly source is worth having

In [5]:
if records is not None and not records.empty:
    total = records[records["market"] == "total"]
    full_years = total.groupby("year")["month"].count()
    full_years = full_years[full_years == 12].index
    seasonal = total[total["year"].isin(full_years)]

    if not seasonal.empty:
        shape = seasonal.groupby("month")["value_usd_mn"].mean()
        ax = shape.plot(marker="o", title="Average monthly apparel exports (complete years only)")
        ax.set_xlabel("Month")
        ax.set_ylabel("USD millions")
        ax.set_xticks(range(1, 13))
        plt.tight_layout()
        plt.show()
        print(f"based on {len(full_years)} complete years")
    else:
        print("no complete years to average")
else:
    print("skipped — no data")

skipped — no data


## 8. US and UK share of the total

In [6]:
if records is not None and not records.empty:
    by_market = records.pivot_table(
        index="year", columns="market", values="value_usd_mn", aggfunc="sum"
    )
    keep = [c for c in ("total", "us", "uk") if c in by_market.columns]
    shares = by_market[keep].copy()
    for market in ("us", "uk"):
        if market in shares.columns and "total" in shares.columns:
            shares[f"{market}_share_pct"] = 100 * shares[market] / shares["total"]
    display(shares.tail(10).round(1))
else:
    print("skipped — no data")

skipped — no data


## 9. How this is used alongside EDB

JAAF and EDB both report Sri Lankan apparel exports, and they **do not agree**,
because they are not measuring the same thing:

- **JAAF** — "apparel & textiles" as a whole.
- **EDB** — a finer product tree with `Apparel` as one sub-category among 17.

CeyNex shows both **side by side and never averages them**. The scope
difference is stated in the answer's assumptions. Averaging two figures with
different scopes produces a number neither organisation would defend.